---
title: PLACEHOLDER
subtitle: Imputation results across all contigs
date: "9999-12-31"
edit_url: null
---

In [ ]:
import altair as alt
from harpy.report.components import StatsBox, print_html, standard_itable
from harpy.report.utilities import binned_histogram
from harpy.report.theme import get_okabe
import itables
import os
import pandas as pd
import sys
itables.init_notebook_mode(connected=True)


In [ ]:
basedir ="paramset/reports/data/impute.compare.stats"
model = "diploid"
usebx = 'true'
bxlimit = 50000
k = 15
s = 20
ngen = 5
extra = "None"


In [ ]:
#| label: placeholder_imputemodelstats
(
    StatsBox()
    .add(model, "Model", textcol = "#a998cc")
    .add(str(usebx), "usebx")
    .add(k, "K Parameter")
    .add(s, "S Parameter")
    .add(ngen, "ngen Parameter")
    .add(bxlimit, "bxlimit")
).render()


In [ ]:
#| label: placeholder_imputeextraparams

print_html(f"Extra Parameters: {extra}")


:::{card} Model Parameters

![](#placeholder_imputemodelstats)
![](#placeholder_imputeextraparams)
:::

In [ ]:
compare = os.path.join(basedir, "impute.compare.stats")
info = os.path.join(basedir, "impute.infoscore")

gcts = pd.read_table(compare, sep = "\t", comment = "#", header=None)
if len(gcts) == 0:
    print_html(f"Stats file ({os.path.relpath(compare)}) is empty. Perhaps there were no biallelelic SNPs that bcftools identified.")
    sys.exit(0)
gcts = gcts[[1, 22, 24, 23, 26]]
gcts.columns = ["sample", "RefHom", "Het", "AltHom", "missing"]
gcts_long = gcts.melt(id_vars='sample', var_name='Conversion', value_name='Count')

infos = pd.read_table(info, header = None)
if len(infos) == 0:
    print_html(f"Info score file ({os.path.relpath(info)}) is empty. Perhaps there were no SNPs that bcftools identified.")
    sys.exit(0)

infos.columns = ["contig", "position", "info"]


In [ ]:
gcts_minmax = gcts_long[gcts_long['Conversion'] != 'missing']['Count'].agg(['min', 'max']).to_dict()

(
    colored_boxes()
    .add(len(gcts), "Samples", width = 210)
    .add(gcts_minmax['min'], "Min Imputations", width = 210)
    .add(gcts_minmax['max'], "Max Imputations", width = 210)
    .add(infos['info'].mean(), "Mean INFO Score", width = 210)
    .add(infos['info'].std(), "Std Dev INFO Score", width = 210)
).render()


## Imputation Quality
`STITCH` outputs a novel `INFO/INFO_SCORE` tag in the resulting VCF files that represents the
"quality" of the imputation. This value ranges from `0` to `1.0`, with `0` being
very poor, and `1.0` being very good. You will want to filter this file to remove
genotypes with a low `INFO_SCORE`, which can be done with `bcftools`. In the charts below,
the hover tooltip will inform you how many SNPs will be removed when filtering at that `INFO-SCORE` threshold.
Here is an example snippet of keeping loci with an `INFO_SCORE` of at least `0.2`: 

```bash
# use -Ob for bcf output (smaller file size)

bcftools view -Ob -i 'INFO/INFO_SCORE >= 0.2' file.vcf > filtered.file.bcf
```


### Global Quality Distribution
This is the global frequency distribution of `INFO_SCORE` values for
the variants in the VCF file across all contigs. This value ranges from `0` (poor) to `1.0` (good).

In [ ]:
_hist = binned_histogram(infos['info'], 0.1, normalize = True).iloc[:-1]
_hist['cumulative'] = _hist['proportion'].cumsum()
(
    alt.Chart(_hist)
    .mark_area(interpolate = "monotone")
    .encode(
        x=alt.X('interval:O', title='INFO_SCORE', bin = 'binned').axis(labelAngle=0),
        y=alt.Y('proportion:Q', title = "Percent SNPs").axis(format = '%'),
        tooltip = [
            alt.Tooltip('interval:O', title = "INFO_SCORE", bin="binned"),
            alt.Tooltip('proportion:Q', title = "Percent of SNPs").format('.2%'),
            alt.Tooltip('cumulative:Q', title = "SNPs filtered at this score").format('.2%')
        ]
    )
    .properties(
        title=alt.Title('Imputation Quality', subtitle = "Distribution across all contigs"),
        usermeta={'embedOptions': {'downloadFileName': f'infoscore.global'}}
    )
)


### Per-Contig Imputation Quality
This is the per-contig frequency distribution of `INFO_SCORE` values for the variants in the VCF file, up to 40 contigs with the greatest SNP densities. The `INFO_SCORE` value ranges from `0` (poor) to `1.0` (good).

In [ ]:
# keep the 40 largest contigs or use the ones provided by the user
_keep = infos['contig'].value_counts().head(40).index.tolist()

# find the biggest block length of the contigs to be kept to create the bin intervals
df_cont = infos[infos['contig'].isin(_keep)][['contig', 'info']].reset_index(drop = True)
#_largest  = df_cont.groupby('contig')['block_length'].max().nlargest(1).index[0]

df_cont['row_num'] = df_cont.groupby('contig').cumcount()
wide_df = df_cont.pivot(index='row_num', columns='contig', values='info').reset_index(drop=True)
del df_cont

# Create the histogram bins for the contig with the largest haplo blocks
_hist = binned_histogram(wide_df[_keep[0]], 0.1, normalize = True).drop(columns = ['proportion']).iloc[:-1]

# iterate through the contigs and left-join to the bin/interval columns
for i in wide_df.columns:
    _binned = binned_histogram(wide_df[i], 0.1, normalize = True).drop(columns = ['interval']).rename(columns = {'proportion': i}).iloc[:-1]
    _hist = _hist.merge(_binned, on = 'bin')
del wide_df


In [ ]:
dropdown = alt.binding_select(options=_keep, name='Contig: ')
ycol_param = alt.param(value=_keep[0], bind=dropdown)

(
    alt.Chart(_hist)
    .mark_area(interpolate = "monotone")
    .transform_calculate(y=f'datum[{ycol_param.name}]')
    .transform_window(ycuml="sum(y)")
    .encode(
        x=alt.X('bin:Q', bin = 'binned', title = "INFO_SCORE").axis(labelAngle=0),
        y=alt.Y('y:Q').title('Percent SNPs').axis(format = '%'),
        tooltip = [
            alt.Tooltip('interval:N', title = "INFO_SCORE"),
            alt.Tooltip('y:Q', title = 'Percent SNPs', format = '.2%'),
            alt.Tooltip('ycuml:Q', title = "SNPs filtered at this score").format('.2%')
        ]
    )
    .add_params(ycol_param)
    .properties(
        title= alt.Title("Imputation Quality", subtitle = "Distribution across a single contig"),
        usermeta={'embedOptions': {'downloadFileName': 'infoscore.contig'}}
    )
)


## Individual Statistics
### Imputations Per Sample
This chart visualizes the count and proportion of `missing` genotypes per sample imputed
into other genotypes. Following standard VCF convention:

**Ref**: the reference allele (from the reference genome)

**Alt**: the alternative allele

**Missing**: how many SNPs remain missing (failed to impute) for that individual

As an example, the `Homozygous Ref` is the number of missing genotypes that were imputed into
a homozygous genotype for the reference allele. The table on the other tab is the
underlying data used to create the visualization.

::::{tab-set}
:::{tab-item} Plot
![](#placeholder_indiv_impute_stats)
:::
:::{tab-item} Table
![](#placeholder_indiv_impute_stats_table)
:::
::::

In [ ]:
colnames = ",".join(gcts.columns)
d_rename = {
    "sample": "Sample",
    "AltHom": "Homozygous Alt",
    "Het": "Heterozygous",
    "RefHom": "Homozygous Ref",
    "missing": "Missing"
}
for k,v in d_rename.items():
    colnames = colnames.replace(k,v)
gcts.columns = colnames.split(",")


In [ ]:
#| label: placeholder_indiv_impute_stats

hover = alt.selection_point(
    fields=['Sample'],
    on='pointerover',  
    nearest=False,   
    empty=False
)

stroke_color = (
    alt.when(hover).then(alt.value("magenta"))
    .otherwise(alt.value(None))
)

(
    alt.Chart(gcts)
    .transform_fold(list(gcts.columns)[1:], as_ = ['Metric', 'Count'])
    .transform_calculate(random="random()")
    .mark_point(size = 65)
    .encode(
        x=alt.X('Count:Q').axis(labelAngle = -40),
        y=alt.Y('Metric:N', title = None),
        yOffset='random:Q',
        color='Metric:N',
        stroke=stroke_color,
        tooltip = [
            'Sample:N',
            alt.Tooltip('Metric:N', title = "Variant"),
            alt.Tooltip('Count:Q', format = ',')
        ]
    )
    .configure_legend(orient = "top")
    .add_params(hover)
    .properties(
        title=alt.Title('Individual Imputation Performance', subtitle = "Genotypes refer to imputation outcome"),
        usermeta={'embedOptions': {'downloadFileName': f'samples.imputestats'}}
    )
).interactive()


###

In [ ]:
#| label: placeholder_indiv_impute_stats_table

standard_itable(gcts, filename = "impute.indiv.stats", fixedcols = 1)
